In [1]:
from __future__ import annotations
from pathlib import Path
from typing import Iterable, Mapping, Optional
import h5py as h5
import numpy as np

In [11]:
def write_xdmf_for_h5(
    h5_path: str | Path,
    keys: Iterable[str],
    *,
    grid_key: str = "volume",
    spacing_xyz: tuple[float, float, float] = (1.0, 1.0, 1.0),
    origin_xyz: tuple[float, float, float] = (0.0, 0.0, 0.0),
    center: str = "Cell",
    make_vec_vds_if_needed: bool = True,
    vds_suffix: str = "_zyx3",
    attribute_type_override: Optional[Mapping[str, str]] = None,
    coord_group: str = "_xdmf_coords",
) -> Path:
    """
    Create an XDMF file referencing selected HDF5 datasets as a true
    RectilinearGrid.

    Assumed scalar layout
    --------------------
    (Z, Y, X)

    Supported vector layouts
    ------------------------
    (Z, Y, X, 3)
    (3, Z, Y, X)   -> converted via VDS to (Z, Y, X, 3)

    Parameters
    ----------
    spacing_xyz : (dx, dy, dz)
        Physical voxel spacing in XYZ coordinates.
    origin_xyz : (ox, oy, oz)
        Physical origin in XYZ coordinates.
    center : {"Node", "Cell"}
        Whether the attribute arrays are point-centered or cell-centered.

        - Node: data shape must match node counts
        - Cell: data shape (Z, Y, X) is interpreted as cell data, and
          coordinate arrays have length (X+1, Y+1, Z+1)
    coord_group : str
        HDF5 group used to store coordinate arrays for XDMF.
    """

    h5_path = Path(h5_path)
    if not h5_path.exists():
        raise FileNotFoundError(h5_path)

    if center not in {"Node", "Cell"}:
        raise ValueError("center must be 'Node' or 'Cell'")

    keys = list(keys)
    attribute_type_override = dict(attribute_type_override or {})

    dx, dy, dz = map(float, spacing_xyz)
    ox, oy, oz = map(float, origin_xyz)

    with h5.File(h5_path, "a" if make_vec_vds_if_needed else "r") as f:
        if grid_key not in f:
            raise KeyError(f"Missing grid_key '{grid_key}'")

        grid = f[grid_key]
        if not isinstance(grid, h5.Dataset):
            raise TypeError(f"'{grid_key}' is not an h5.Dataset")

        if grid.ndim < 3:
            raise ValueError(f"'{grid_key}' must have at least 3 dims, got {grid.shape}")

        # spatial shape always interpreted as (Z, Y, X)
        Z, Y, X = grid.shape[-3:]

        # topology dimensions in XDMF order are Z Y X
        if center == "Cell":
            topo_Z, topo_Y, topo_X = Z + 1, Y + 1, X + 1
        else:
            topo_Z, topo_Y, topo_X = Z, Y, X

        topo_dims = f"{topo_Z} {topo_Y} {topo_X}"

        # create explicit rectilinear coordinates
        # GeometryType="VXVYVZ" expects X, Y, Z arrays
        if center == "Cell":
            x_coords = ox + dx * np.arange(X + 1, dtype=np.float64)
            y_coords = oy + dy * np.arange(Y + 1, dtype=np.float64)
            z_coords = oz + dz * np.arange(Z + 1, dtype=np.float64)
        else:
            x_coords = ox + dx * np.arange(X, dtype=np.float64)
            y_coords = oy + dy * np.arange(Y, dtype=np.float64)
            z_coords = oz + dz * np.arange(Z, dtype=np.float64)

        grp = f.require_group(coord_group)

        def _write_or_replace(name: str, arr: np.ndarray) -> str:
            path = f"{coord_group}/{name}"
            if name in grp:
                del grp[name]
            grp.create_dataset(name, data=arr, dtype=np.float64)
            return path

        x_path = _write_or_replace("x", x_coords)
        y_path = _write_or_replace("y", y_coords)
        z_path = _write_or_replace("z", z_coords)

        attr_blocks = []

        for key in keys:
            if key not in f:
                raise KeyError(f"Missing dataset '{key}'")

            ds = f[key]
            if not isinstance(ds, h5.Dataset):
                raise TypeError(f"'{key}' is not an h5.Dataset")

            target_key = key
            atype = attribute_type_override.get(key)

            if ds.ndim == 3 and ds.shape == (Z, Y, X):
                atype = atype or "Scalar"
                dims = f"{Z} {Y} {X}"

            elif ds.ndim == 4 and ds.shape == (Z, Y, X, 3):
                atype = atype or "Vector"
                dims = f"{Z} {Y} {X} 3"

            elif ds.ndim == 4 and ds.shape[0] == 3 and ds.shape[1:] == (Z, Y, X):
                atype = atype or "Vector"

                if not make_vec_vds_if_needed:
                    raise ValueError(f"{key} is (3, Z, Y, X) but VDS creation is disabled.")

                vds_key = f"{key}{vds_suffix}"
                if vds_key in f:
                    del f[vds_key]

                layout = h5.VirtualLayout(shape=(Z, Y, X, 3), dtype=ds.dtype)
                vsrc = h5.VirtualSource(ds)
                for c in range(3):
                    layout[:, :, :, c] = vsrc[c, :, :, :]

                f.create_virtual_dataset(vds_key, layout)
                target_key = vds_key
                dims = f"{Z} {Y} {X} 3"

            else:
                raise ValueError(
                    f"Unsupported shape for '{key}': {ds.shape}. "
                    f"Expected (Z,Y,X), (Z,Y,X,3), or (3,Z,Y,X) with "
                    f"(Z,Y,X)=({Z},{Y},{X})."
                )

            attr_blocks.append(
                f"""      <Attribute Name="{key}" AttributeType="{atype}" Center="{center}">
        <DataItem Dimensions="{dims}" Format="HDF">{h5_path.name}:/{target_key}</DataItem>
      </Attribute>"""
            )

    xmf_path = h5_path.with_suffix(".xdmf")

    xml = f"""<?xml version="1.0" encoding="utf-8"?>
<Xdmf Version="3.0">
  <Domain>
    <Grid Name="RectilinearGrid" GridType="Uniform">
      <Topology TopologyType="3DRectMesh" Dimensions="{topo_dims}"/>
      <Geometry GeometryType="VXVYVZ">
        <DataItem Dimensions="{topo_X}" NumberType="Float" Precision="8" Format="HDF">{h5_path.name}:/{x_path}</DataItem>
        <DataItem Dimensions="{topo_Y}" NumberType="Float" Precision="8" Format="HDF">{h5_path.name}:/{y_path}</DataItem>
        <DataItem Dimensions="{topo_Z}" NumberType="Float" Precision="8" Format="HDF">{h5_path.name}:/{z_path}</DataItem>
      </Geometry>
{chr(10).join(attr_blocks)}
    </Grid>
  </Domain>
</Xdmf>
"""
    xmf_path.write_text(xml, encoding="utf-8")
    return xmf_path

In [9]:
h5_file = Path("../data/285_uCT_0915-1830.vol.vol_.labels.vec.down2.h5")

In [13]:
_ = write_xdmf_for_h5(h5_file,keys=["vol","vol_","labels","vec"],grid_key="vol")

In [2]:
from structure_tensor.h5_io import load_h5_datasets

In [3]:

file_   = Path("../data/285_uCT_0915-1830.vol.labels.h5")
file    = Path("../data/285_uCT_0915-1830.vec.h5")




dict1 = load_h5_datasets(file_,keys=["vol","vol_","labels"])

dict2 = load_h5_datasets(file,keys=["vec"])

Available keys: ['_xdmf_coords', 'labels', 'vol', 'vol_']
Available keys: ['_xdmf_coords', 'vec', 'vec_zyx3']


In [4]:
for key, item in dict1.items():
    print(f"{key}, {item.dtype}")

for key, item in dict2.items():
    print(f"{key}, {item.dtype}")

vol, float32
vol_, float32
labels, float32
vec, float32


In [7]:
file_out = Path("../data/285_uCT_0915-1830.vol.vol_.labels.vec.h5")

with h5.File(file_out,"w") as F:
    
    F.create_dataset("vol",data=dict1["vol"].astype(np.uint8),compression="gzip")
    F.create_dataset("vol_",data=dict1["vol_"].astype(np.uint16),compression="gzip")
    F.create_dataset("labels",data=dict1["labels"].astype(np.uint8),compression="gzip")

    F.create_dataset("vec",data=dict2["vec"].astype(np.float32),compression="gzip")

In [8]:

file_out = Path("../data/285_uCT_0915-1830.vol.vol_.labels.vec.down2.h5")

with h5.File(file_out, "w") as F:

    F.create_dataset(
        "vol",
        data=dict1["vol"][::2, ::2, ::2].astype(np.uint8),
        compression="gzip"
    )

    F.create_dataset(
        "vol_",
        data=dict1["vol_"][::2, ::2, ::2].astype(np.uint16),
        compression="gzip"
    )

    F.create_dataset(
        "labels",
        data=dict1["labels"][::2, ::2, ::2].astype(np.uint8),
        compression="gzip"
    )

    # vector field (assuming shape [3, z, y, x] OR [z, y, x, 3])
    vec = dict2["vec"]

    if vec.shape[0] == 3:  # [3, z, y, x]
        vec_ds = vec[:, ::2, ::2, ::2]
    else:  # [z, y, x, 3]
        vec_ds = vec[::2, ::2, ::2, :]

    F.create_dataset(
        "vec",
        data=vec_ds.astype(np.float32),
        compression="gzip"
    )